# RAG arm -- Qwen2.5-Coder inference with retrieved few-shot examples + retrieved schema

Companion notebook to `mongodb_nl_to_sql_1.ipynb` (the baseline, full-schema arm). Everything about model loading, generation parameters, and output post-processing is **identical** to that notebook on purpose -- the only thing that differs between the two arms is prompt *content* (retrieved few-shot examples + one database's schema here, vs. the full 6-database schema there). That keeps the comparison isolated to what retrieval adds, not confounded by a different model config.

**Before running this notebook**, the retrieval step must already have produced `rag/data/rag_prompts.json` -- run these locally first (CPU only, no GPU needed):
```
python rag/build_split.py
python rag/build_retrieval_index.py
python rag/build_prompts.py
```
then commit/push so this notebook's `git clone` picks up `rag/data/rag_prompts.json`.


In [ ]:
!git clone --branch evaluation-pipeline https://github.com/tarun1125/CSAIML-Capstone-Project-20.git


In [ ]:
%pip install transformers accelerate torch bitsandbytes sentencepiece pymongo huggingface_hub codebleu bert-score tree-sitter tree-sitter-python


In [ ]:
import torch
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

model.eval()


In [ ]:
cd /content/CSAIML-Capstone-Project-20


In [ ]:
import os

from google.colab import userdata
HF_TOKEN   = userdata.get("HF_TOKEN")

import logging
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger("capstone-eval-rag")


## RAG inference

Loads `rag/data/rag_prompts.json` (21 held-out test cases, each with a `system_prompt` already assembled by `rag/build_prompts.py` from retrieved few-shot examples + the retrieved database's schema). No prompt construction happens in this cell -- retrieval already happened locally; this cell's only job is calling Qwen and saving raw output, same as the baseline notebook's cell 5.


In [ ]:
import json
import time
from pathlib import Path

# -----------------------------------------------------------------------------
# Load the 21 pre-built RAG prompts (produced locally by rag/build_prompts.py,
# committed to the repo, pulled in by the git clone in cell 0). Each entry
# already carries its own system_prompt -- this cell does zero prompt
# construction, only generation, so behavior here matches cell 5 of the
# baseline notebook exactly aside from where the prompt came from.
# -----------------------------------------------------------------------------
ROOT = Path.cwd()
if not (ROOT / "rag" / "data" / "rag_prompts.json").exists():
    ROOT = ROOT.parent
PROMPTS_FILE = ROOT / "rag" / "data" / "rag_prompts.json"

if not PROMPTS_FILE.exists():
    raise FileNotFoundError(
        f"{PROMPTS_FILE} not found -- run rag/build_split.py, "
        f"rag/build_retrieval_index.py, and rag/build_prompts.py locally first, "
        f"then commit+push rag/data/rag_prompts.json before running this notebook."
    )

with open(PROMPTS_FILE, encoding="utf-8") as f:
    CASES = json.load(f)

log.info("Loaded %d RAG test cases from %s", len(CASES), PROMPTS_FILE)

predictions_rag = []

for i, case in enumerate(CASES, 1):
    qid = case["id"]
    nl = case["question"]
    database = case.get("gold_database")
    system_prompt = case["system_prompt"]

    log.info("[%d/%d] [%s] querying Qwen (RAG arm, predicted_db=%s, db_match=%s)...",
             i, len(CASES), qid, case.get("predicted_database"), case.get("database_match"))
    t0 = time.time()

    try:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": nl}
        ]

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.0,
            do_sample=False
        )

        generated = outputs[0][inputs.input_ids.shape[1]:]
        raw = tokenizer.decode(generated, skip_special_tokens=True).strip()
        raw = raw.replace("```python", "").replace("```", "").strip()

        latency = time.time() - t0
        log.info("[%s] OK (%.2fs): %s", qid, latency, raw[:80])

        predictions_rag.append({
            "id": qid,
            "question": nl,
            "database": database,
            "generated_query": raw,
            "latency_s": round(latency, 3),
            "predicted_database": case.get("predicted_database"),
            "database_match": case.get("database_match"),
            "retrieved_ids": case.get("retrieved_ids"),
        })

    except Exception as e:
        log.error("[%s] FAILED: %s", qid, e)
        predictions_rag.append({
            "id": qid,
            "question": nl,
            "database": database,
            "generated_query": "",
            "error": str(e)
        })

# Save to rag/data/qwen_rag_results.json -- same record shape (id/question/
# database/generated_query) as data/qwen2.5-coder_results.json, so the
# existing normalize.py works unchanged:
#   python normalize.py rag/data/qwen_rag_results.json rag/data/qwen_rag_normalized.json
OUT_PATH = ROOT / "rag" / "data" / "qwen_rag_results.json"
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions_rag, f, indent=2)

log.info("Saved %d predictions -> %s", len(predictions_rag), OUT_PATH)


## Next steps (run locally, not in Colab)

Download `rag/data/qwen_rag_results.json` from this Colab session (or `git pull` it back locally if you push it from here), then:
```
python normalize.py rag/data/qwen_rag_results.json rag/data/qwen_rag_normalized.json
python rag/score_rag.py
```
`rag/score_rag.py` needs a real Atlas connection so it runs locally, same as `execute_gold.py` / `evaluation/execute_queries.py` already do. It prints RAG's accuracy next to the baseline's accuracy recomputed on the *same* 21 held-out ids -- that same-slice comparison is the number that actually answers whether retrieval helped.
